```markdown
# Agentes com Web Open-Source: Criando LLMs com Acesso à Internet 🌐🤖

### Resumo
Este notebook apresenta o conceito e a implementação prática de **Agentes de IA**, demonstrando como conectar um modelo de linguagem open-source de última geração à internet para realizar buscas em tempo real. Você aprenderá a configurar o modelo **Qwen**, preparar as ferramentas de busca na Wikipedia e na Web, e entender como dar "superpoderes" de informação atualizada para uma inteligência artificial. Este é o ponto de partida ideal para quem deseja sair de respostas estáticas e criar assistentes verdadeiramente inteligentes e conectados.

---

## 🎯 Objetivos de Aprendizagem
Ao final deste estudo, você será capaz de:
1. **Compreender o conceito de Agentes e Ferramentas (Tools):** Entender como uma LLM interage com sistemas externos.
2. **Configurar e Carregar Modelos Open-Source de Grande Porte (LLMs):** Inicializar o modelo Qwen no ambiente gratuito do Colab utilizando quantização e otimizações de hardware.
3. **Implementar Ferramentas de Busca:** Criar funções em Python para recuperar informações atualizadas da Wikipedia e do buscador DuckDuckGo.
4. **Interpretar Templates de Chat (Chat Templates):** Compreender como formatar diálogos estruturados para que o modelo responda de forma consistente.
```

```markdown
## 🛠️ Pré-requisitos e Bibliotecas Utilizadas

Para que o nosso agente funcione, precisamos instalar e carregar algumas ferramentas essenciais. Abaixo explicamos o motivo de cada uma:

* **`transformers` (Hugging Face):** É a biblioteca principal para trabalhar com modelos de Deep Learning pré-treinados. Ela nos permite baixar, carregar e interagir com o modelo de linguagem **Qwen** de forma simples.
* **`accelerate`:** Uma biblioteca da Hugging Face que gerencia de forma inteligente a memória do sistema (RAM e VRAM da GPU), permitindo carregar modelos pesados em hardwares limitados como o Colab gratuito.
* **`ddgs` (DuckDuckGo Search):** Uma biblioteca leve que permite fazer buscas de forma programática na web através do buscador DuckDuckGo, simulando o que um usuário faria no navegador.
* **`torch` (PyTorch):** O motor matemático de Inteligência Artificial por trás das nossas redes neurais. Ele processará os tensores (estruturas matemáticas de dados) necessários para gerar texto.
* **`requests`:** Uma biblioteca padrão do Python usada para fazer requisições HTTP, essencial para acessarmos a API pública da Wikipedia.
```

In [3]:
!pip install -q -U "transformers>=5.10.1" accelerate ddgs

```markdown
### Explicação: Instalação de Dependências (Célula acima)

* **O que este passo faz:** Instala na máquina virtual do Colab as versões necessárias das bibliotecas `transformers` (garantindo que seja superior ou igual à versão 5.10.1), `accelerate` e o pacote `ddgs` para buscas na web.
* **Por que ele é necessário:** Por padrão, o ambiente do Google Colab não vem com essas bibliotecas específicas instaladas (ou vem com versões desatualizadas). Precisamos delas para carregar o modelo de IA e realizar as pesquisas.
* **Como interpretar o resultado:** Ao rodar, você verá linhas de progresso de download e instalação. O parâmetro `-q` (quiet) esconde o excesso de logs para deixar seu notebook mais limpo, e `-U` garante a atualização para as versões estáveis mais recentes.
```

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

```markdown
### Explicação: Importação dos Módulos Core (Célula acima)

* **O que este passo faz:** Importa a biblioteca gráfica de tensores `torch` e as duas classes principais que usaremos do ecossistema Hugging Face:
  1. `AutoTokenizer`: Responsável por traduzir texto humano em números (tokens) que o modelo de IA entende.
  2. `AutoModelForCausalLM`: A estrutura da própria rede neural treinada para prever a próxima palavra (Causal Language Modeling).
* **Por que ele é necessário:** Sem importar estes módulos, o Python não reconhecerá as funções necessárias para baixar e executar a inteligência artificial nas células seguintes.
```

In [5]:
MODEL_ID = "Qwen/Qwen3-4B"

tokenizar = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

```markdown
### Explicação: Inicialização do Modelo e Tokenizador (Célula acima)

* **O que este passo faz:** Define qual modelo usaremos (`Qwen/Qwen3-4B`), baixa os arquivos de configuração do Hugging Face Hub, inicializa o tokenizador correspondente e carrega o modelo neural diretamente na memória.
* **Por que ele é escrito assim:**
  - Utilizamos `dtype=torch.float16` para carregar o modelo usando precisão de 16-bits (meia precisão) em vez de 32-bits. Isso corta o consumo de memória RAM/VRAM pela metade sem perda significativa de qualidade de resposta.
  - O parâmetro `device_map="auto"` delega para a biblioteca `accelerate` a tarefa de decidir onde colocar cada parte do modelo (preferencialmente na GPU do Colab, se disponível, para aceleração de hardware).
* **Como interpretar o resultado:** Você verá barras de download dos pesos do modelo (que possuem alguns gigabytes). Se aparecer um aviso sobre `HF_TOKEN`, não se preocupe: o modelo Qwen é de acesso público e não exige autenticação obrigatória para download básico.

> 💡 **Para pensar:** O identificador `Qwen/Qwen3-4B` refere-se à arquitetura Qwen3 com aproximadamente 4 bilhões de parâmetros. Se o ambiente do Colab desconectar por falta de memória RAM de GPU, pode ser interessante explorar versões menores como de 1.5B ou técnicas adicionais de quantização (como 8-bit ou 4-bit) usando a biblioteca `bitsandbytes`.
```

In [6]:
tokenizer = tokenizar # corrigir apenas o nome da varivel, para não recarregar tudo

```markdown
### Explicação: Ajuste de Variáveis (Célula acima)

* **O que este passo faz:** Atribui o objeto instanciado incorretamente como `tokenizar` para o nome correto em inglês padrão `tokenizer`.
* **Por que ele é necessário:** Na célula de carregamento anterior, a variável foi declarada com um pequeno erro de digitação (`tokenizar`). Para evitar ter que baixar novamente os gigabytes do modelo e gastar tempo de processamento reexecutando a célula pesada, fazemos esse redirecionamento rápido de referência em memória.
```

In [14]:
# apply_chat_template: transforma a lista de mensagens no formato de texto que o Qwen3 espera.
# enable_thinking=False: o Qwen3 tem um modo em que ele "pensa em voz alta" antes de responder. Desligado, a resposta fica curta, o que é melhor para o agente.
# saida[0][entradas.input_ids.shape[1]:]: corta o prompt e mantém só o texto novo.

messagens = [{"role": "user", "content": "Quem foi Carl Sagan? Responda em uma frase"}]

entradas = tokenizer.apply_chat_template(
    messagens,
    add_generation_prompt=True,
    enable_thinking=False,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

saida = model.generate(**entradas, max_new_tokens=100)
resposta = tokenizer.decode(saida[0][entradas["input_ids"].shape[1]:], skip_special_tokens=True)
print(resposta)

Carl Sagan foi um astrofísico, escritor e cosmólogo americano conhecido por sua contribuição ao estudo do universo e por sua popularização da ciência através de obras como "A Via Láctea".


```markdown
### Explicação: Executando nossa Primeira Inferência (Célula acima)

* **O que este passo faz:** Envia uma pergunta simples para a IA para validar se ela está carregada corretamente e funcionando no idioma português.
* **Por que ele é escrito assim:**
  - `apply_chat_template`: Modelos de chat modernos não recebem texto corrido pura e simplesmente. Eles são treinados com estruturas especiais que delimitam onde começa a fala do usuário (`user`) e onde deve começar a resposta do assistente (`assistant`). Essa função automatiza essa formatação.
  - `enable_thinking=False`: Desativa o raciocínio interno estendido (cadeia de pensamento) do modelo, ideal para quando precisamos de uma resposta direta e concisa.
  - `saida[0][entradas["input_ids"].shape[1]:]`: Os modelos de linguagem geram a resposta repetindo o prompt original enviado. Esta linha faz um fatiamento (slice) no vetor de saída para remover a nossa pergunta, deixando apenas os caracteres novos gerados pelo modelo.
* **Como interpretar o resultado:** O modelo deve imprimir na tela uma frase curta, objetiva e correta sobre a identidade e legado de Carl Sagan.
```

In [16]:
# Agente - Buscas wikipedia
import requests
from ddgs import DDGS

def busca_wikipedia(consulta, idioma="pt", max_resultados=3):
    try:
        resp = requests.get(
            f"https://{idioma}.wikipedia.org/w/api.php",
            params={
                "action": "query", "generator": "search", "gsrsearch": consulta,
                "gsrlimit": max_resultados, "prop": "extracts", "exintro": 1,
                "explaintext": 1, "exlimit": max_resultados, "format": "json",
            },
            headers={"User-Agent": "estudos-agente/0.1 (projeto de estudo)"},
            timeout=10,
        )
        resp.raise_for_status()
        paginas = resp.json().get("query", {}).get("pages", {})
    except Exception as erro:
        return f"ERRO NA BUSCA: {erro}"
    if not paginas:
        return "Nenhum resultado encontrado."
    ordenadas = sorted(paginas.values(), key=lambda p: p.get("index", 0))
    return "\n".join(f"- {p['title']}: {p.get('extract', '')[:500]}" for p in ordenadas)

```markdown
### Explicação: Criação da Ferramenta Wikipedia (Célula acima)

* **O que este passo faz:** Define uma função em Python (`busca_wikipedia`) que faz uma varredura na API oficial do portal Wikipedia baseado em um termo de pesquisa, trazendo os resumos iniciais dos artigos mais relevantes.
* **Por que ele é necessário:** Modelos de linguagem sofrem de um fenômeno chamado **Alucinação** (inventar fatos com convicção) e possuem uma data limite de conhecimento (knowledge cutoff). Conectar a IA a APIs confiáveis como a Wikipedia garante que ela tenha acesso a dados históricos e enciclopédicos precisos.
* **Como interpretar o resultado:** A função encapsula toda a complexidade de uma consulta web, faz o tratamento de erros (`try/except`) para evitar que o código quebre o sistema do agente se a internet oscilar, e formata os dados limpos em uma string estruturada.
```

In [17]:
print(busca_wikipedia("Quem foi Carl Sagan?"))

- Carl Sagan: Carl Edward Sagan (Nova Iorque, 9 de novembro de 1934 — Seattle, 20 de dezembro de 1996) foi um cientista planetário, astrônomo, astrobiólogo, astrofísico, escritor, divulgador científico e ativista norte-americano. Sagan é autor de mais de 600 publicações científicas e também de mais de vinte livros de ciência e ficção científica.
Foi durante a vida um grande defensor do ceticismo e do uso do método científico. Promoveu a busca por inteligência extraterrestre através do projeto SETI e instituiu
- Pálido Ponto Azul: Pálido Ponto Azul (em inglês: Pale Blue Dot) é uma fotografia da Terra tirada em 14 de fevereiro de 1990 pela sonda Voyager 1, de uma distância de seis bilhões de quilômetros (40,5 AU) da Terra, como parte de uma série de imagens do Sistema Solar denominada Retrato de Família.
Nessa fotografia, o tamanho aparente da Terra é menor do que um pixel; o planeta aparece como um pequeno ponto na imensidão do espaço, no meio de um raio solar captado pela lente da câme

```markdown
### Explicação: Teste de Ferramenta Wikipedia (Célula acima)

* **O que este passo faz:** Executa a função criada anteriormente enviando o termo de pesquisa "Quem foi Carl Sagan?" e imprime o retorno na tela.
* **Como interpretar o resultado:** Você verá uma lista formatada com marcadores contendo títulos de artigos correspondentes (ex: Carl Sagan, Pálido Ponto Azul) seguidos de seus respectivos parágrafos de introdução diretamente extraídos da enciclopédia real.
```

In [18]:
# Agente - Buscas na Web
def busca_web(consulta, max_resultados=3):
    try:
        resultados = DDGS(timeout=10).text(consulta, max_results=max_resultados)
    except Exception as erro:
        return f"ERRO NA BUSCA: {erro}"
    linhas = [f"- {r['title']}: {r['body']}" for r in resultados]
    return "\n".join(linhas) if linhas else "Nenhum resultado encontrado."

```markdown
### Explicação: Criação da Ferramenta de Busca Web Geral (Célula acima)

* **O que este passo faz:** Define a função `busca_web` que utiliza a biblioteca DuckDuckGo Search para buscar tópicos gerais em qualquer site da internet.
* **Por que ele é necessário:** Enquanto a Wikipedia é ótima para fatos históricos consolidáveis, ela não possui notícias de última hora, preços de produtos, previsão do tempo ou discussões recentes. Essa ferramenta abre as portas para toda a World Wide Web.
* **Como interpretar o resultado:** A função retorna uma lista estruturada de títulos de páginas e pequenos trechos (snippets) extraídos das páginas de resultados da busca.
```

In [22]:
print(busca_web("Quem foi Carl Sagan?"))

- Carl Sagan - Wikipedia: Carl Edward Sagan (/ ˈseɪɡən /; SAY-gən; November 9, 1934 – December 20, 1996) [4] was an American astronomer, planetary scientist and science communicator.
- Quem foi Carl Sagan, o cientista que desvendou o cosmos e ...: Quem foi Carl Sagan, o cientista que desvendou o cosmos e combateu a desinformação No dia 20 de dezembro de 1996 o mundo se despedia de um dos cientistas mais populares que já teve. Ele foi das universidades à televisão, sendo pioneiro na divulgação científica até os dias de hoje.
- Quem foi Carl Sagan? - Olhar Digital: May 8, 2025 · Carl Sagan foi um dos maiores divulgadores científicos do século XX. Conheça sua trajetória e legado cultural.


```markdown
### Explicação: Teste de Ferramenta Web Geral (Célula acima)

* **O que este passo faz:** Valida nossa ferramenta de busca na web trazendo referências externas que complementam a pesquisa bibliográfica do nosso exemplo.
* **Como interpretar o resultado:** Exibe referências vindas de portais de notícias e portais de divulgação científica atualizados sobre a história de Carl Sagan.

---

## ⚠️ Erros Comuns e Dicas de Solução

1. **Erro de Memória (Out of Memory - OOM):** Se o ambiente do Colab travar e reiniciar ao carregar o modelo, verifique se você ativou o acelerador de hardware por GPU (Ambiente de Execução -> Alterar tipo de ambiente de execução -> Selecionar T4 GPU).
2. **Erros de Conexão na Busca:** A biblioteca `ddgs` realiza raspagem de dados e buscas automatizadas. Se você rodar a função de busca muitas vezes seguidas em um curto período, o DuckDuckGo pode bloquear temporariamente seu endereço de IP por segurança (Rate Limiting). Se isso acontecer, aguarde alguns minutos antes de tentar novamente.
3. **Warning do Hugging Face Hub:** O aviso em vermelho sugerindo configurar um `HF_TOKEN` é apenas informativo. Para modelos abertos como o Qwen, você pode ignorar esse aviso com segurança.

---

## 🎓 Resumo do que Aprendemos

* **Modelos Open-Source Locais:** Aprendemos a carregar e interagir com modelos de linguagem no nosso próprio ecossistema utilizando técnicas para otimização de memória (como precisão `float16`).
* **Sistemas Baseados em Ferramentas:** Entendemos que um agente não precisa guardar todo o conhecimento do mundo em seus parâmetros neurais. Podemos estender sua capacidade fornecendo funções Python que atuam como "olhos e ouvidos" no mundo externo.
* **APIs vs Web Scraping:** Comparamos o acesso a dados estruturados de enciclopédias (Wikipedia API) com buscas dinâmicas em tempo real pela internet geral (DuckDuckGo).

---

## 📝 Exercícios Propostos

1. **[Fácil] Customização de Ferramentas:** Modifique a função `busca_wikipedia` para aceitar um parâmetro opcional chamado `limite_caracteres` que mude o tamanho do resumo retornado por cada artigo (atualmente fixado em `500` no código original).
2. **[Médio] Nova Ferramenta - Clima:** Crie uma nova ferramenta chamada `obter_clima(cidade)` que faça uma consulta de clima atualizada para uma cidade fornecida (você pode usar uma API pública gratuita como a *Open-Meteo* ou simular uma busca web focada no clima).
3. **[Desafiador] O Loop de Decisão do Agente:** Escreva uma lógica manual simples (usando condicionais `if/else` simples) onde a LLM analisa a pergunta do usuário e decide se precisa chamar a `busca_wikipedia`, a `busca_web` ou se pode responder diretamente de cabeça, simulando manualmente o mecanismo de raciocínio de um agente inteligente.

---

## 🚀 Para Ir Além
* **ReAct Pattern (Reasoning and Acting):** Pesquise sobre como os agentes modernos usam um loop de pensamento, decisão de ação, execução de ferramenta e observação para resolver tarefas complexas.
* **LangChain & LlamaIndex:** Conheça os frameworks líderes de mercado que automatizam toda essa criação de agentes e conexões de ferramentas que criamos manualmente neste notebook.
* **Quantização de Modelos (GGUF / AWQ):** Estude como converter modelos pesados para rodarem de forma extremamente rápida até mesmo em celulares ou computadores modestos.
```